In [2]:
%pip install -q numpy numba google-cloud-storage google-auth

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import sys

REPO_URL = "https://github.com/payamdav/pycrypto.git"
REPO_NAME = "pycrypto"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

sys.path.insert(0, os.path.join(REPO_PATH, "packages/tools/google_cloud_storage_tools"))

import time
import io
import numpy as np
from gcs_tools import read_file, gcs_json_key_file

gcs_json_key_file()

'/home/payam/projects/pycrypto/notebooks/studies/lookback_lookahead_features_labels/gcp_service_account_key.json'

In [6]:
GCS_BUCKET = "payamdprojectbucket"
GCS_KEY    = "lookback_lookahead_data"

t0 = time.perf_counter()
data_bytes = read_file(GCS_BUCKET, GCS_KEY)
elapsed = time.perf_counter() - t0

candles = np.load(io.BytesIO(data_bytes))

file_size = len(data_bytes)
print(f"Loaded from gs://{GCS_BUCKET}/{GCS_KEY}")
print(f"File size : {file_size / 1024 / 1024:.2f} MB  ({file_size:,} bytes)")
print(f"Load time : {elapsed:.4f}s")
print(f"Array shape: {candles.shape}  dtype: {candles.dtype}")

Loaded from gs://payamdprojectbucket/lookback_lookahead_data
File size : 95.45 MB  (100,085,984 bytes)
Load time : 55.0725s
Array shape: (1042561, 12)  dtype: float64


In [7]:
LOOK_BACK  = 1440
LOOK_AHEAD = 240
K          = 100

In [ ]:
from packages.asset_analyzer import asset_snapshot_lookback_lookahead_normalize_prepare_all

t0 = time.perf_counter()
fl = asset_snapshot_lookback_lookahead_normalize_prepare_all(candles, LOOK_BACK, LOOK_AHEAD, K)
elapsed = time.perf_counter() - t0
print(f"asset_snapshot_lookback_lookahead_normalize_prepare_all elapsed: {elapsed:.4f}s  shape: {fl.shape}  size: {fl.nbytes / 1024 / 1024:.2f} MB")

In [ ]:
from gcs_tools import write_file

GCS_BUCKET = "payamdprojectbucket"
GCS_KEY    = "fl_data"

buf = io.BytesIO()
np.save(buf, fl)
buf.seek(0)

t0 = time.perf_counter()
write_file(GCS_BUCKET, GCS_KEY, buf)
elapsed = time.perf_counter() - t0
print(f"Written to gs://{GCS_BUCKET}/{GCS_KEY}  elapsed: {elapsed:.4f}s")

In [ ]:
%pip install -q matplotlib

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Index layout from normalize function
# l_e_vwap:         indices 52–55  (left-anchored expanding, step=60, count=4)
# l_e_n_imbalances: indices 56–59
L_E_VWAP_SI         = 52
L_E_N_IMBALANCES_SI = 56

bins = np.arange(-1.0, 1.05, 0.1)  # bandwidth = 0.1, 20 bins over [-1, 1]

horizon_labels = ["1 h (60 min)", "2 h (120 min)", "3 h (180 min)", "4 h (240 min)"]
colors_vwap    = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
colors_imb     = ["#CCB974", "#64B5CD", "#E07B54", "#76A865"]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle("Label Distributions — Normalized VWAP & Volume Imbalance\n(left-anchored expanding windows, step = 60 min)", fontsize=14, fontweight="bold", y=1.01)

for j in range(4):
    # ── Row 0: l_e_vwap ──────────────────────────────────────────────
    ax = axes[0, j]
    col = fl[:, L_E_VWAP_SI + j]
    ax.hist(col, bins=bins, color=colors_vwap[j], edgecolor="white", linewidth=0.4, density=True)
    ax.set_title(f"VWAP Label — {horizon_labels[j]}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Normalized VWAP (clipped to [−1, 1])", fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.set_xlim(-1.05, 1.05)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(0.5))
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(0.1))
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.grid(axis="x", linestyle=":", alpha=0.3)
    mean_v, std_v = col.mean(), col.std()
    ax.axvline(mean_v, color="black", linestyle="--", linewidth=1.2, label=f"mean={mean_v:.3f}")
    ax.legend(fontsize=8)

    # ── Row 1: l_e_n_imbalances ──────────────────────────────────────
    ax = axes[1, j]
    col = fl[:, L_E_N_IMBALANCES_SI + j]
    ax.hist(col, bins=bins, color=colors_imb[j], edgecolor="white", linewidth=0.4, density=True)
    ax.set_title(f"Imbalance Label — {horizon_labels[j]}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Normalized Imbalance (clipped to [−1, 1])", fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.set_xlim(-1.05, 1.05)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(0.5))
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(0.1))
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.grid(axis="x", linestyle=":", alpha=0.3)
    mean_v = col.mean()
    ax.axvline(mean_v, color="black", linestyle="--", linewidth=1.2, label=f"mean={mean_v:.3f}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()